In [1]:
import os
import json
import pandas as pd
from datetime import datetime
from dotenv import load_dotenv

from options import OptionSurface, Deribit, OKX, Bybit
from portfolio_management import Portfolio
from api_client import TradingDeskAPI
from scanner import MarketScanner

In [ ]:
# Initialize all classes and parameters

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
FRACTION = os.getenv("FRACTION")
MAX_POSITION = os.getenv("MAX_POSITION")
ENTRY_EV_THRESHOLD = os.getenv("ENTRY_EV_THRESHOLD") # require 1% edge, default = 0
EXIT_EV_THRESHOLD = os.getenv("EXIT_EV_THRESHOLD")
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]
# currencies = ["BTC", "ETH", "HYPE", "SOL", "ZEC"]

https://alphasignal-dev.moretoncp.com


In [ ]:
# 1. Get all dfs
orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet")
equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

# fills_df = fills_df.iloc[0:0]

In [24]:
# 2. Get newly executed trades
fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)
fills_df

,order_id,condition_id,token_id,outcome,side,price,shares,fee_rate_bps,timestamp,fill_id
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,BUY,0.981,4.87,0.0,2026-08-14 08:25:11+00:00,4c3c68e2-7b89-463c-ae03-48b92808eeef


In [10]:
# 3. Reconstruct portfolio
# 4. Calculate realized P&L
positions_df, realized_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old
realized_df

,condition_id,token_id,outcome,realized_shares,realized_pnl,realized_fees
0,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,0.0,0.0,0.0


In [11]:
# 5. Sync positions with api
portfolio.reconcile_positions(api=api, positions_df=positions_df)

POSITIONS SYNCED


In [12]:
# 6. Get latest order state
orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)
orders_df

,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,NO,buy,0.981,4.877934,GTC,OPEN,2026-08-14 09:28:01.978131+00:00,NaT


In [13]:
# 7. Mark positions to market
positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)
positions_df

,condition_id,token_id,outcome,shares,cost_basis,avg_entry_price,realized_pnl,realized_shares,realized_fees,current_price,market_value,unrealized_pnl,unrealized_return
0,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,No,4.87,4.77747,0.981,0.0,0.0,0.0,0.985,4.79695,0.01948,0.004077


In [14]:
# 8. Calculate equity
equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
                                realized_df=realized_df, equity_df=equity_df)
equity_df

Equity:  100.01
Return:  0.0131%
Sharpe: N/A
Sortino: N/A


,timestamp,cash,market_value,equity,realized_pnl,unrealized_pnl,daily_return,period_return,sharpe,sortino
0,2026-08-14 07:47:24.047885+00:00,100.00000,0.00000,100.00000,0.0,0.00000,NaN,None,NaN,NaN
1,2026-08-15 14:14:05.272655+00:00,95.21618,4.79695,100.01313,0.0,0.01948,NaN,0.000131,NaN,NaN


In [15]:
# 9. Initialize variance surface
deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
okx = OKX(currencies=currencies, target_expiry=target_expiry)
bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

spot: 63048.0 volume24h: 1.9996
spot: 1882.7 volume24h: 13.8911
spot: 63040.9 volume24h: 1937.72974071
spot: 1883.12 volume24h: 29829.973884
spot: 63035.9 volume24h: 3971.899523
spot: 1883.12 volume24h: 27701.52774


In [16]:
# 10. Get current markets
# markets = api.get_markets(limit=10000, liquidity_num_min=10000, volume_num_min=5000) # old api
# all_markets_df = pd.DataFrame(markets)

all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)
all_markets_df.head()

Fetched 100 markets | Total: 100
Fetched 100 markets | Total: 200
Fetched 100 markets | Total: 300
Fetched 100 markets | Total: 400
Fetched 100 markets | Total: 500
Fetched 100 markets | Total: 600
Fetched 100 markets | Total: 700
Fetched 100 markets | Total: 800
Fetched 100 markets | Total: 900
Fetched 100 markets | Total: 1,000


,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,negRiskMarketID,oneDayPriceChange,seriesColor,showGmpSeries,showGmpOutcome,umaResolutionStatus,oneHourPriceChange,eventStartTime,gameStartTime,groupItemRange
0,559651,Xi Jinping out before 2027?,0xa467b14d51f01b957109d9cbb1d6c124fab2a089d52e...,xi-jinping-out-before-2027,,2026-12-31T00:00:00Z,255668.90297,2025-07-03T20:37:00.228Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,NaN,NaN,NaN,None,None,NaN,NaN,NaN,NaN,NaN
1,559652,Will Gavin Newsom win the 2028 Democratic pres...,0x0f49db97f71c68b1e42a6d16e3de93d85dbf7d4148e3...,will-gavin-newsom-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,371457.03958,2025-07-11T18:35:56.805Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,-0.008,,False,False,NaN,NaN,NaN,NaN,NaN
2,559653,Will Alexandria Ocasio-Cortez win the 2028 Dem...,0xe6bcc2f1dd025ce5e1833190f7c60a71171c94f805df...,will-alexandria-ocasio-cortez-win-the-2028-dem...,,2028-11-07T00:00:00Z,437459.92072,2025-07-11T18:35:59.075Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,-0.024,,False,False,NaN,NaN,NaN,NaN,NaN
3,559654,Will Pete Buttigieg win the 2028 Democratic pr...,0x4c325469d9b516ef4e6b8f73a81a12607dec075e3c2f...,will-pete-buttigieg-win-the-2028-democratic-pr...,,2028-11-07T00:00:00Z,454592.31471,2025-07-11T18:35:58.818Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,NaN,,False,False,NaN,NaN,NaN,NaN,NaN
4,559655,Will Josh Shapiro win the 2028 Democratic pres...,0xd65891729ce093cc12236856837eba1a0872fc7998fd...,will-josh-shapiro-win-the-2028-democratic-pres...,,2028-11-07T00:00:00Z,430084.17586,2025-07-11T18:36:01.098Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0x2c3d7e0eee6f058be3006baabf0d54a07da254ba47fe...,NaN,,False,False,NaN,NaN,NaN,NaN,NaN


In [17]:
# 11. Scan markets
markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)
opportunities_df.head()

iv 0.07330828953123085
p_touch_above: 1.0 p_touch_below: 0.0

BTC
buy_yes_ev: 0.96689727
sell_yes_ev: -0.97297113
buy_no_ev: -0.97297113
sell_no_ev: 0.9668972699999999
buy_yes_kelly: 0.5
sell_yes_kelly: -17.998738571016844
buy_no_kelly: -17.998738571016855
sell_no_kelly: 0.5
iv 0.5843791417764115
p_touch_above: 0.0006 p_touch_below: 1.0

BTC
buy_yes_ev: -0.015434249999999998
sell_yes_ev: 0.012433719999999999
buy_no_ev: 0.012433720000000025
sell_no_ev: -0.015434250000000026
buy_yes_kelly: -0.007842879693729177
sell_yes_kelly: 0.4769827800505151
buy_no_kelly: 0.4769827800505151
sell_no_kelly: -0.007842879693729191
iv 0.5780910872579328
p_touch_above: 0.00095 p_touch_below: 1.0

BTC
buy_yes_ev: -0.023622969999999997
sell_yes_ev: 0.019543879999999996
buy_no_ev: 0.01954387999999996
sell_no_ev: -0.023622970000000024
buy_yes_kelly: -0.012109040078579736
sell_yes_kelly: 0.4768223489158714
buy_no_kelly: 0.4768223489158714
sell_no_kelly: -0.01210904007857975
iv 0.5743358365182031
p_touch_above: 

,id,question,conditionId,slug,resolutionSource,endDate,liquidity,startDate,image,icon,...,buy_yes_ev,sell_yes_ev,buy_no_ev,sell_no_ev,buy_yes_kelly,sell_yes_kelly,buy_no_kelly,sell_no_kelly,best_ev,best_action
166,573656,"Will Bitcoin hit $150k by December 31, 2026?",0x02deb9538f5c123373adaa4ee6217b01745f1662bc90...,will-bitcoin-hit-150k-by-december-31-2026,,2027-01-01T05:00:00Z,188955.98603,2025-08-07T16:29:32.879Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.966897,-0.972971,-0.972971,0.966897,0.500000,-17.998739,-17.998739,0.500000,0.966897,buy_yes_ev
761,701552,"Will Ethereum dip to $1,500 by December 31, 2026?",0xcf25ccc8360f3952139fb5e3f14cb9bb0636cb53d0e3...,will-ethereum-dip-to-1500-by-december-31-2026-...,,2027-01-01T05:00:00Z,105649.65637,2025-11-24T19:27:14.681Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.112011,-0.147058,-0.147058,0.112011,0.099130,-0.183830,-0.183830,0.099130,0.112011,buy_yes_ev
762,701553,"Will Ethereum dip to $1,000 by December 31, 2026?",0xacb33346b59a2a3770e2391b7d1b0e77d8dcdcf840a6...,will-ethereum-dip-to-1000-by-december-31-2026-...,,2027-01-01T05:00:00Z,107228.5736,2025-11-24T19:27:19.011Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.047337,-0.070490,-0.070490,0.047337,0.026800,-0.376147,-0.376147,0.026800,0.047337,sell_no_ev
760,701549,"Will Ethereum reach $3,500 by December 31, 2026?",0x42945ea5657d6f6e77969a06661b29d6f6295083d1ef...,will-ethereum-reach-3500-by-december-31-2026,,2027-01-01T05:00:00Z,54550.872,2025-11-24T19:27:16.702Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,-0.059780,0.037747,0.037747,-0.059780,-0.033445,0.223973,0.223973,-0.033445,0.037747,sell_yes_ev
748,701503,"Will Bitcoin dip to $35,000 by December 31, 2026?",0x2745c38ff0617cb345c1d2df19b4f74ea777508e0741...,will-bitcoin-dip-to-35000-by-december-31-2026-...,,2027-01-01T05:00:00Z,130875.7839,2025-11-24T19:07:19.763Z,https://polymarket-upload.s3.us-east-2.amazona...,https://polymarket-upload.s3.us-east-2.amazona...,...,0.033960,-0.055993,-0.055993,0.033960,0.019000,-0.332236,-0.332236,0.019000,0.033960,sell_no_ev


In [18]:
# 12. manage cancel orders
orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)
orders_df

,order_id,condition_id,token_id,outcome,side,price,requested_size,order_type,status,created_at,cancelled_at
0,0x8039083a81a0749075a21658567ad6e0e449ea2ac99e...,0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...,9699347185440015640867052761315094444335927219...,NO,buy,0.981,4.877934,GTC,OPEN,2026-08-14 09:28:01.978131+00:00,NaT


In [19]:
# 13. Risk management
orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

pos: condition_id         0xac32e73aa9e0dae801d88d4f81efd2ef3fa0f04b815f...
token_id             9699347185440015640867052761315094444335927219...
outcome                                                             No
shares                                                            4.87
cost_basis                                                     4.77747
avg_entry_price                                                  0.981
realized_pnl                                                       0.0
realized_shares                                                    0.0
realized_fees                                                      0.0
current_price                                                    0.985
market_value                                                   4.79695
unrealized_pnl                                                 0.01948
unrealized_return                                             0.004077
Name: 0, dtype: object
market: id                                       

KeyError: 'question'

In [ ]:
# 14. New opportunities
orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

In [ ]:
# 15. Safe dfs
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
portfolio.save(DATA_DIR, orders_df, "orders")
portfolio.save(DATA_DIR, fills_df, "fills")
portfolio.save(DATA_DIR, positions_df, "positions")
portfolio.save(DATA_DIR, realized_df, "realized_pnl")
portfolio.save(DATA_DIR, equity_df, "equity")

In [23]:
portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")

ValueError: Can't infer object conversion type: 166    [{'id': '36173', 'ticker': 'when-will-bitcoin-...
735    [{'id': '89502', 'ticker': 'what-price-will-bi...
736    [{'id': '89502', 'ticker': 'what-price-will-bi...
737    [{'id': '89502', 'ticker': 'what-price-will-bi...
738    [{'id': '89502', 'ticker': 'what-price-will-bi...
739    [{'id': '89502', 'ticker': 'what-price-will-bi...
740    [{'id': '89502', 'ticker': 'what-price-will-bi...
741    [{'id': '89502', 'ticker': 'what-price-will-bi...
742    [{'id': '89502', 'ticker': 'what-price-will-bi...
743    [{'id': '89502', 'ticker': 'what-price-will-bi...
744    [{'id': '89502', 'ticker': 'what-price-will-bi...
745    [{'id': '89502', 'ticker': 'what-price-will-bi...
746    [{'id': '89502', 'ticker': 'what-price-will-bi...
747    [{'id': '89502', 'ticker': 'what-price-will-bi...
748    [{'id': '89502', 'ticker': 'what-price-will-bi...
749    [{'id': '89502', 'ticker': 'what-price-will-bi...
750    [{'id': '89519', 'ticker': 'what-

In [ ]:
ENTRY_EV_THRESHOLD = 0.01   # require 1% edge, default = 0
EXIT_EV_THRESHOLD = 0.02
FRACTION = 0.25
MAX_POSITION = 0.05

load_dotenv(r"D:/OneDrive/Trading/Prediction Markets/Moreton Capital/.env")
BASE_URL = os.getenv("BASE_URL")
USER_EMAIL = os.getenv("USER_EMAIL")
USER_PASSWORD = os.getenv("USER_PASSWORD")
JWT_TOKEN = os.getenv("JWT")
DATA_DIR = os.getenv("DATA_DIR")
print(BASE_URL)

with open(f"{DATA_DIR}/crypto_tag_ids.json", "r") as f:
    crypto_tag_ids = set(json.load(f))

s = OptionSurface()
portfolio = Portfolio()
api = TradingDeskAPI(base_url=BASE_URL, email=USER_EMAIL, password=USER_PASSWORD, token=JWT_TOKEN)
scanner = MarketScanner(api=api)

target_expiry_str = "25DEC26"
target_expiry = datetime.strptime(target_expiry_str, "%d%b%y").strftime("%Y%m%d")

currencies = ["BTC", "ETH"]

for i in range(1):
    # 1. Get all dfs
    orders_df = pd.read_parquet(f"{DATA_DIR}/orders.parquet")
    fills_df = pd.read_parquet(f"{DATA_DIR}/fills.parquet")
    positions_df = pd.read_parquet(f"{DATA_DIR}/positions.parquet")
    realized_pnl_df = pd.read_parquet(f"{DATA_DIR}/realized_pnl.parquet") # old pnl df
    equity_df = pd.read_parquet(f"{DATA_DIR}/equity.parquet")

    # 2. Get newly executed trades
    fills_df = portfolio.sync_fills(api=api, fills_df=fills_df)

    # 3. Reconstruct portfolio
    # 4. Calculate realized P&L
    positions_df, realized_df = portfolio.reconstruct_positions_fifo(fills_df=fills_df) # new realized_df to overwrite old

    # 5. Sync positions with api
    portfolio.reconcile_positions(api=api, positions_df=positions_df)

    # 6. Get latest order state
    orders_df = portfolio.sync_orders(api=api, orders_df=orders_df)

    # 7. Mark positions to market
    positions_df = portfolio.mark_positions_to_market(api=api, positions_df=positions_df)

    # 8. Calculate equity
    equity_df = portfolio.calculate_equity(api=api, positions_df=positions_df, 
                                    realized_df=realized_df, equity_df=equity_df)

    # 9. Initialize variance surface
    deribit = Deribit(currencies=currencies, target_expiry=target_expiry)
    okx = OKX(currencies=currencies, target_expiry=target_expiry)
    bybit = Bybit(currencies=currencies, target_expiry=target_expiry)

    s.initialize(currencies=currencies, exchanges=[deribit, okx, bybit])

    # 10. Get current markets
    all_markets_df = api.get_all_markets(DATA_DIR=DATA_DIR, count_limit=10, liquidity_num_min=10000, volume_num_min=5000)

    # 11. Scan markets
    markets_df, opportunities_df = scanner.scan_market(markets_df=all_markets_df, s=s, crypto_tag_ids=crypto_tag_ids)

    # 12. manage cancel orders
    orders_df = portfolio.manage_open_orders(api=api, orders_df=orders_df, markets_df=markets_df, 
                ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

    # 13. Risk management
    orders_df = portfolio.run_risk_management(api=api, positions_df=positions_df, 
            markets_df=markets_df, orders_df=orders_df, EXIT_EV_THRESHOLD=EXIT_EV_THRESHOLD)

    # 14. New opportunities
    orders_df = portfolio.run_new_opportunities(api=api, opportunities_df=opportunities_df, orders_df=orders_df, 
            ENTRY_EV_THRESHOLD=ENTRY_EV_THRESHOLD, MAX_POSITION=MAX_POSITION, FRACTION=FRACTION)

    # 15. Safe dfs
    portfolio.save_snapshots(DATA_DIR, markets_df, "markets")
    portfolio.save_snapshots(DATA_DIR, opportunities_df, "opportunities")
    portfolio.save(DATA_DIR, orders_df, "orders")
    portfolio.save(DATA_DIR, fills_df, "fills")
    portfolio.save(DATA_DIR, positions_df, "positions")
    portfolio.save(DATA_DIR, realized_df, "realized_pnl")
    portfolio.save(DATA_DIR, equity_df, "equity")

    # time.sleep(300)

In [ ]:
# Polymarket    Model (exchanges: deribit, bybit, okx)    Edge

# 80k Dec-26        22%          20%       +2%
# 90k Dec-26        14%          12%       +2%
# 100k Dec-26        9.5%         5.9%     +3.6%
# 110k Dec-26        7%           4%       +3%
# 120k Dec-26        5%           2%       +3%

In [ ]:
# Keep trading journal

# Your thesis.
# Why you think the market is mispriced.
# Position size.
# Exit criteria.
# What actually happened.

In [ ]:
# Stage 2 — Semi-automated execution

# The bot does:

# find opportunities
# calculate size
# prepare orders

# You approve:

# BUY YES
# Market: BTC above $150k
# Price: 0.43
# Size: $500
# Expected edge: +8%

# Click "confirm".

# This is useful because prediction markets can have:

# ambiguous wording
# resolution risks
# sudden news events

In [ ]:
# Day 1 — API connection + market ingestion

# Goal:

# Can I pull markets automatically?

# Build:

# get_markets()

# Output:

# {
#  "condition_id": "...",
#  "question": "Will X happen?",
#  "tokens": [
#     {
#       "token_id": "YES",
#       "price": 0.42
#     },
#     {
#       "token_id": "NO",
#       "price": 0.58
#     }
#  ]
# }

# Store:

# markets

# condition_id
# question
# yes_token
# no_token
# created_time
# Day 2 — Historical snapshots

# Goal:

# Can I reconstruct what the market looked like yesterday?

# Every 5 minutes:

# while True:

#     markets = api.get_markets()

#     for m in markets:
#         database.save_snapshot(m)

#     sleep(300)

# Database:

# market_snapshots

# timestamp
# condition_id
# yes_price
# no_price
# volume
# Day 3 — Order book collection

# Now collect microstructure data.

# For selected markets:

# get_orderbook(token_id)

# Store:

# orderbook_snapshots

# timestamp

# token_id

# best_bid
# best_ask

# bid_depth
# ask_depth

# Calculate:

# Spread
# spread=ask−bid

# Example:

# Bid:
# 0.42

# Ask:
# 0.46

# Spread:
# 4 cents
# Day 4 — Build your scanner

# Your first scanner should be dumb but useful.

# Signal 1: Large moves
# if abs(price_change_24h) > 0.10:
#     flag()

# Example:

# AI model release

# Yesterday:
# 35%

# Today:
# 52%

# Move:
# +17%
# Signal 2: Liquidity opportunities
# if spread > 0.08:
#     flag()
# Signal 3: Volume spikes
# if volume_today > 5 * average_volume:
#     flag()

# Your output:

# TOP MARKETS TO REVIEW

# 1.
# Question:
# Will Fed cut rates?

# Price:
# 42%

# 24h move:
# +12%

# Reason:
# Large movement


# 2.
# Question:
# Will Company X acquire Y?

# Price:
# 33%

# Spread:
# 11 cents

# Reason:
# Wide market
# Day 5 — Add your probability workflow

# Do NOT automate this yet.

# Create a manual table:

# trade_journal.csv

# market,current_price,my_probability,edge,reason
# Fed cut,0.42,0.55,0.13,"Inflation falling"
# AI launch,0.35,0.45,0.10,"Company comments"

# The key question:

# Market probability:
# 42%

# My probability:
# 55%

# Difference:
# +13%
# Day 6 — Paper trading

# Before your C++ engine touches anything:

# Create:

# paper_buy(
#     market,
#     price,
#     size
# )

# Track:

# Position:
# YES Fed cut

# Entry:
# 42c

# Size:
# $100

# Current:
# 48c

# P&L:
# +$14
# Day 7 — Analytics

# Calculate:

# Return
# profit / capital
# Drawdown

# Largest loss from peak.

# Sortino

# Track:

# returns
# negative returns only

# Your output:

# Paper Portfolio

# Trades:
# 18

# Win rate:
# 61%

# Return:
# +8.4%

# Max drawdown:
# -2.1%

# Sortino:
# 2.4